In [ ]:
# ============================================================
# Print prompt templates only
# Shows only 2 batch samples per prompt
# Also prints few-shot examples
# Update path to actual MAHED task 1 path
# ============================================================

import os
import pandas as pd
from pathlib import Path
from typing import List, Tuple

# -------------------------
# Configuration
# -------------------------
SEED = 42

VALID_LABELS = ["hate", "hope", "not_applicable"]

TRAIN_PATH = "/content/drive/MyDrive/496/MAHED Sub Task 1/MAHED Sub Task 1/train.csv"
VAL_PATH = "/content/drive/MyDrive/496/MAHED Sub Task 1/MAHED Sub Task 1/validation.csv"

OUTPUT_DIR = "/content/drive/MyDrive/496/MAHED_Gemini_Task3"
SUBSET_PATH = f"{OUTPUT_DIR}/mahed_val_subset_500.csv"

# Separate few-shot files so you can inspect 1-shot and 3-shot prompts clearly
FEW_SHOT_1_PATH = f"{OUTPUT_DIR}/few_shot_examples_1_per_class.csv"
FEW_SHOT_3_PATH = f"{OUTPUT_DIR}/few_shot_examples_3_per_class.csv"

PROMPT_TYPES_TO_PRINT = [
    "zero_shot_definition",
    "zero_shot_nli",
    "zero_shot_distinction",
    "few_shot_definition",
    "three_shot_cultural",
    "super_prompt",
]


# ============================================================
# Data helpers
# ============================================================
def clean_label_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower()


def load_mahed_data(train_path: str = TRAIN_PATH, val_path: str = VAL_PATH) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)

    train_df["text"] = train_df["text"].astype(str)
    val_df["text"] = val_df["text"].astype(str)

    train_df["label"] = clean_label_series(train_df["label"])
    val_df["label"] = clean_label_series(val_df["label"])

    train_df = train_df[train_df["label"].isin(VALID_LABELS)].reset_index(drop=True)
    val_df = val_df[val_df["label"].isin(VALID_LABELS)].reset_index(drop=True)

    return train_df, val_df


def load_or_create_subset_preview(val_df: pd.DataFrame) -> pd.DataFrame:
    """
    Loads your existing 500-sample subset if it exists.
    If not, uses the first validation rows only for prompt preview.
    """
    if os.path.exists(SUBSET_PATH):
        subset_df = pd.read_csv(SUBSET_PATH)
        print(f"Loaded subset from: {SUBSET_PATH}")
    else:
        subset_df = val_df.copy().reset_index(drop=True)
        print("Subset file not found. Using validation dataframe for preview only.")

    return subset_df


def load_or_create_few_shot_examples(
    train_df: pd.DataFrame,
    n_per_class: int,
    save_path: str,
) -> pd.DataFrame:
    """
    Creates fixed few-shot examples if they do not already exist.
    """
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    if os.path.exists(save_path):
        examples_df = pd.read_csv(save_path)
        print(f"Loaded few-shot examples from: {save_path}")
        return examples_df

    parts = []
    for label in VALID_LABELS:
        class_df = train_df[train_df["label"] == label]
        examples = class_df.sample(n=n_per_class, random_state=SEED)
        parts.append(examples[["text", "label"]])

    examples_df = pd.concat(parts).reset_index(drop=True)
    examples_df.to_csv(save_path, index=False)

    print(f"Saved few-shot examples to: {save_path}")
    return examples_df


# ============================================================
# Prompt helpers
# ============================================================
def format_numbered_texts(texts: List[str]) -> str:
    return "\n".join([f"{i + 1}. {str(text).strip()}" for i, text in enumerate(texts)])


def format_few_shot_examples(examples_df: pd.DataFrame) -> str:
    lines = []

    for _, row in examples_df.iterrows():
        lines.append(f'Text: {str(row["text"]).strip()}')
        lines.append(f'Label: {str(row["label"]).strip()}')
        lines.append("")

    return "\n".join(lines).strip()


def build_batch_prompt(
    texts: List[str],
    prompt_type: str,
    few_shot_examples: pd.DataFrame | None = None,
) -> str:
    numbered_texts = format_numbered_texts(texts)

    definition_labels = """Labels:
hate = hateful, hostile, abusive, or harmful toward a person or group
hope = supportive, encouraging, empathetic, or constructive
not_applicable = neither hate nor hope, or unclear/neutral"""

    nli_labels = """Labels:
hate = the text entails hateful, hostile, abusive, or harmful meaning toward a person or group
hope = the text entails supportive, encouraging, empathetic, or constructive meaning
not_applicable = neither hate nor hope is entailed, or the meaning is unclear/neutral"""

    distinction_labels = """Use these distinctions:
hate = attacks, dehumanizes, insults, threatens, or promotes hostility toward a person or group
hope = expresses support, empathy, encouragement, solidarity, or positive action
toxic_or_offensive_but_not_hate = rude, aggressive, insulting, or offensive, but not clearly hate and not clearly hope
not_applicable = neutral, unclear, irrelevant, or does not fit hate or hope

Important rule:
If the text is only rude, offensive, political criticism, or general anger, do not label it hate unless it clearly targets a person or group with harmful or hateful meaning."""

    cultural_labels = """Labels:
hate = hateful, hostile, abusive, or harmful toward a person or group
hope = supportive, encouraging, empathetic, or constructive
not_applicable = neither hate nor hope, or unclear/neutral

Consider Arabic cultural context, dialectal wording, religious expressions, political language, sarcasm, and indirect meaning. Do not classify a text as hate only because it contains negative words. Use the intended meaning of the full text."""

    super_labels = """Labels:
hate = the text clearly attacks, dehumanizes, insults, threatens, or promotes hostility toward a person or group.
hope = the text clearly expresses support, empathy, encouragement, solidarity, help, relief, or constructive positive action.
not_applicable = the text is neutral, unclear, irrelevant, ambiguous, only emotional, only poetic, only romantic, only descriptive, or does not clearly fit hate or hope."""

    super_rules = """Decision rules:
1. Use the intended meaning of the full Arabic text, not isolated keywords.
2. Consider Arabic cultural context, dialectal wording, religious expressions, political language, sarcasm, indirect meaning, code mixing, and Arabizi.
3. Do not classify a text as hate only because it contains negative words, insults, anger, offensive wording, political criticism, or references to conflict.
4. If the text is rude, toxic, offensive, or politically critical but does not clearly attack a person or group with hateful or harmful meaning, label it not_applicable.
5. Do not classify a text as hope only because it contains positive words, love words, poetry, praise, religious phrases, or emotional language.
6. Label hope only when the text clearly expresses support, encouragement, empathy, solidarity, help, relief, or constructive positive action.
7. When the text is ambiguous between a real label and not_applicable, choose not_applicable.
8. When both hate and hope seem possible, choose the label that is more central to the intended meaning of the full text."""

    output_rule = """Return exactly one line per text using this format:
1: hate
2: hope
3: not_applicable

Use only these labels: hate, hope, not_applicable.
Do not explain."""

    if prompt_type == "zero_shot_definition":
        return f"""Classify each Arabic text into one label only.

{definition_labels}

{output_rule}

Texts:
{numbered_texts}""".strip()

    if prompt_type == "zero_shot_nli":
        return f"""Classify each Arabic text using an NLI-style decision.

For each text, decide which label is best entailed by the meaning of the text.

{nli_labels}

{output_rule}

Texts:
{numbered_texts}""".strip()

    if prompt_type == "zero_shot_distinction":
        return f"""Classify each Arabic text into one label only.

{distinction_labels}

Final labels allowed:
hate
hope
not_applicable

Map toxic_or_offensive_but_not_hate to not_applicable.

{output_rule}

Texts:
{numbered_texts}""".strip()

    if prompt_type == "few_shot_definition":
        if few_shot_examples is None or few_shot_examples.empty:
            raise ValueError("few_shot_definition requires few_shot_examples.")

        examples = format_few_shot_examples(few_shot_examples)

        return f"""Classify each Arabic text into one label only.

{definition_labels}

Examples:
{examples}

{output_rule}

Texts:
{numbered_texts}""".strip()

    if prompt_type == "three_shot_cultural":
        if few_shot_examples is None or few_shot_examples.empty:
            raise ValueError("three_shot_cultural requires few_shot_examples.")

        examples = format_few_shot_examples(few_shot_examples)

        return f"""Classify each Arabic text into one label only.

{cultural_labels}

Examples:
{examples}

{output_rule}

Texts:
{numbered_texts}""".strip()

    if prompt_type == "super_prompt":
        if few_shot_examples is None or few_shot_examples.empty:
            raise ValueError("super_prompt requires few_shot_examples.")

        examples = format_few_shot_examples(few_shot_examples)

        return f"""Classify each Arabic text into one label only.

{super_labels}

{super_rules}

Examples:
{examples}

{output_rule}

Texts:
{numbered_texts}""".strip()

    raise ValueError(f"Unknown prompt_type: {prompt_type}")
# ============================================================
# Print prompt templates
# ============================================================
def print_prompt_templates_only():
    train_df, val_df = load_mahed_data()
    subset_df = load_or_create_subset_preview(val_df)

    # Only 2 samples from the batch
    preview_texts = subset_df["text"].astype(str).head(2).tolist()

    # Few-shot examples
    one_shot_examples = load_or_create_few_shot_examples(
        train_df=train_df,
        n_per_class=1,
        save_path=FEW_SHOT_1_PATH,
    )

    three_shot_examples = load_or_create_few_shot_examples(
        train_df=train_df,
        n_per_class=3,
        save_path=FEW_SHOT_3_PATH,
    )

    print("\n" + "=" * 100)
    print("BATCH PREVIEW SAMPLES USED IN ALL PROMPTS")
    print("=" * 100)

    for i, text in enumerate(preview_texts, start=1):
        print(f"\nSample {i}:")
        print(text)

    print("\n" + "=" * 100)
    print("ONE-SHOT EXAMPLES USED FOR few_shot_definition")
    print("=" * 100)
    print(format_few_shot_examples(one_shot_examples))

    print("\n" + "=" * 100)
    print("THREE-SHOT EXAMPLES USED FOR three_shot_cultural")
    print("=" * 100)
    print(format_few_shot_examples(three_shot_examples))
    for prompt_type in PROMPT_TYPES_TO_PRINT:

        if prompt_type == "few_shot_definition":
            examples = one_shot_examples

        elif prompt_type in ["three_shot_cultural", "super_prompt"]:
            examples = three_shot_examples

        else:
            examples = None

        prompt = build_batch_prompt(
            texts=preview_texts,
            prompt_type=prompt_type,
            few_shot_examples=examples,
        )

        print("\n" + "=" * 100)
        print(f"PROMPT TYPE: {prompt_type}")
        print("=" * 100)
        print(prompt)
        print("=" * 100)

print_prompt_templates_only()